# Structural metrics

Observed structural metrics, empirical plausibility, attribute mixing, and degree distributions for the reciprocal LLM-generated networks.


## 0 · Imports & Configuration


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pickle, os, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import networkx as nx
from networkx.algorithms.community import louvain_communities
from networkx.algorithms.community.quality import modularity

from matplotlib import font_manager
from IPython.display import display

BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "code":
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "outputs" / "data"
FIGURE_DIR = BASE_DIR / "outputs" / "figures"
PAPER_IMAGE_DIR = BASE_DIR / "code" / "images"
TABLE_DIR = BASE_DIR / "outputs" / "tables"
FONT_DIR = BASE_DIR / "font"
for folder in (DATA_DIR, FIGURE_DIR, TABLE_DIR, PAPER_IMAGE_DIR):
    folder.mkdir(parents=True, exist_ok=True)


# -- fonts (Barlow if available, else system default) --
_font_dir = str(FONT_DIR)
if os.path.isdir(_font_dir):
    for ff in font_manager.findSystemFonts(fontpaths=[_font_dir]):
        font_manager.fontManager.addfont(ff)
    plt.rcParams['font.family'] = 'Barlow'

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

# -- paths --
INPUT_PKL   = DATA_DIR / 'graphs_sample_100_v3_mutual.pkl'
METRICS_CSV = DATA_DIR / 'micro_metrics_sample_100_v3_mutual.csv'
PLAUSIBILITY_CSV = DATA_DIR / 'structural_plausibility_sample_100_v3_mutual.csv'
PLAUSIBILITY_SUMMARY_CSV = DATA_DIR / 'structural_plausibility_summary_sample_100_v3_mutual.csv'
REAL_RANGES_CSV = DATA_DIR / 'real_network_plausibility_ranges.csv'
RANDOM_SEED = 42

# -- attribute lists --
HOMOPHILY_ATTRS = [
    'profile_gender',
    'profile_racethnicity',
    'profile_educ5',
    'w2presvtwho',
    'profile_relig',
]

# Short labels for column/plot names
ATTR_SHORT = {
    'profile_gender':       'gender',
    'profile_racethnicity': 'racethnicity',
    'profile_educ5':        'educ5',
    'w2presvtwho':          'vote',
    'profile_relig':        'relig',
}

np.random.seed(RANDOM_SEED)


## 1 · Load Graphs


In [ ]:
with open(INPUT_PKL, 'rb') as f:
    graphs = pickle.load(f)

# Drop hidden/checkpoint entries that can be created beside the real model folders.
graphs = {
    model: files
    for model, files in graphs.items()
    if not str(model).startswith('.') and files
}

print('Models loaded:', list(graphs.keys()))
total = sum(len(v) for v in graphs.values())
print(f'Total graphs : {total}')


## 2 · Metric Functions


In [ ]:
# -- helpers --

def gini(values):
    arr = np.array(values, dtype=float)
    if arr.sum() == 0:
        return 0.0
    arr = np.sort(arr)
    n = len(arr)
    idx = np.arange(1, n + 1)
    return (2 * (idx * arr).sum()) / (n * arr.sum()) - (n + 1) / n


def density(G):
    return round(nx.density(G), 6)


def degree_stats(G):
    values = [degree for _, degree in G.degree()]
    return {
        'avg_degree': round(np.mean(values), 4),
        'degree_std': round(np.std(values), 4),
        'degree_max': int(np.max(values)),
        'degree_gini': round(gini(values), 4),
    }


def clustering_stats(G):
    values = list(nx.clustering(G).values())
    return {
        'clustering_mean': round(np.mean(values), 4),
        'clustering_std': round(np.std(values), 4),
        'clustering_zero_frac': round(sum(v == 0 for v in values) / len(values), 4),
    }


def component_stats(G):
    n = G.number_of_nodes()
    lcc = max(nx.connected_components(G), key=len)
    return {
        'lcc_n': len(lcc),
        'lcc_ratio': round(len(lcc) / n, 4),
    }


def avg_path_length(G):
    lcc = max(nx.connected_components(G), key=len)
    sub = G.subgraph(lcc)
    try:
        return round(nx.average_shortest_path_length(sub), 4)
    except Exception:
        return None


def louvain_modularity(G):
    try:
        communities = louvain_communities(G, seed=RANDOM_SEED)
        return round(modularity(G, communities), 4)
    except Exception:
        return None


def attribute_assortativity(G, attr):
    try:
        return round(nx.attribute_assortativity_coefficient(G, attr), 4)
    except Exception:
        return None


def homophily_ratio(G, attr):
    groups = {}
    for node, data in G.nodes(data=True):
        group = data.get(attr)
        if group is not None:
            groups[group] = groups.get(group, 0) + 1
    N = sum(groups.values())
    if N < 2:
        return None
    same_exp = sum(groups[g] * (groups[g] - 1) for g in groups) / (N * (N - 1))
    cross_exp = 1 - same_exp
    if cross_exp == 0 or G.number_of_edges() == 0:
        return None
    cross_obs = sum(
        1 for u, v in G.edges()
        if G.nodes[u].get(attr) != G.nodes[v].get(attr)
    ) / G.number_of_edges()
    return round(cross_obs / cross_exp, 4)


def compute_all_metrics(G):
    row = {'density': density(G)}
    row.update(degree_stats(G))
    row.update(clustering_stats(G))
    row.update(component_stats(G))
    row['avg_path_length'] = avg_path_length(G)
    row['normalized_path_length'] = (
        round(row['avg_path_length'] / np.log(row['lcc_n']), 4)
        if row['avg_path_length'] is not None and row['lcc_n'] > 1
        else None
    )
    row['louvain_modularity'] = louvain_modularity(G)
    for attr in HOMOPHILY_ATTRS:
        short = ATTR_SHORT[attr]
        row[f'{short}_assortativity'] = attribute_assortativity(G, attr)
        row[f'{short}_homophily'] = homophily_ratio(G, attr)
    return row


## 3 · Compute Metrics on LLM Graphs


In [ ]:
records = []

for model, files in graphs.items():
    for fname, G in files.items():
        row = {'model': model, 'file': fname}
        row.update(compute_all_metrics(G))
        records.append(row)
        if len(records) % 100 == 0:
            print(f'  computed observed metrics for {len(records)} graphs')

micro_df = pd.DataFrame(records)

def parse_filename(fname):
    name = fname.replace('.txt', '')
    temp_m  = re.search(r'_temp([0-9.]+)', name)
    chunk_m = re.search(r'_chunk(\d+)', name)
    run_m   = re.search(r'_run(\d+)', name)
    order_m = re.search(r'_run\d+_(.+)$', name)
    return {
        'temp'           : float(temp_m.group(1))  if temp_m  else None,
        'chunked'        : bool(chunk_m),
        'chunk_size'     : int(chunk_m.group(1))   if chunk_m else 0,
        'run'            : int(run_m.group(1))      if run_m   else None,
        'attribute_order': order_m.group(1)         if order_m else None,
    }

micro_df = pd.concat(
    [micro_df, pd.DataFrame(micro_df['file'].apply(parse_filename).tolist())],
    axis=1
)

micro_df.head()


## TABLES S2 and S3


In [ ]:
# ============================================================
# TABLES S2-S3: COMPLETE NETWORK METRICS
# Mean ± SE across 10 runs × 2 attribute orderings
# ============================================================

import numpy as np
import pandas as pd


# Models in manuscript order
MODEL_ORDER = [
    'llama-31-8b-instruct',
    'mistral-7b-instruct-v03',
    'qwen25-7b-instruct',
    'meta_llama_31_70b_instruct_awq_int4',
    'qwen25_72b_instruct_awq',
]

MODEL_LABELS_TABLE = {
    'llama-31-8b-instruct': 'Llama-3.1 8B',
    'mistral-7b-instruct-v03': 'Mistral 7B',
    'qwen25-7b-instruct': 'Qwen2.5 7B',
    'meta_llama_31_70b_instruct_awq_int4': 'Llama-3.1 70B',
    'qwen25_72b_instruct_awq': 'Qwen2.5 72B',
}

METHOD_ORDER = ['c1', 'c5', 'c10', 'c25', 'c50']


def get_method(row):
    if not row['chunked'] or row['chunk_size'] == 0:
        return 'c1'
    return f"c{int(row['chunk_size'])}"


data = micro_df.copy()

data = data[data['model'].isin(MODEL_ORDER)].copy()

data['Model'] = data['model'].map(MODEL_LABELS_TABLE)
data['Method'] = data.apply(get_method, axis=1)

data['Model'] = pd.Categorical(
    data['Model'],
    categories=[MODEL_LABELS_TABLE[m] for m in MODEL_ORDER],
    ordered=True
)

data['Method'] = pd.Categorical(
    data['Method'],
    categories=METHOD_ORDER,
    ordered=True
)


In [ ]:
def make_summary_table(data, metrics):

    grouped = (
        data
        .groupby(
            ['Model', 'temp', 'Method', 'attribute_order'],
            observed=True
        )[metrics]
        .agg(['mean', 'std', 'count'])
    )

    rows = []

    for idx, values in grouped.iterrows():

        model, temp, method, attribute_order = idx

        row = {
            'Model': model,
            'T': temp,
            'Method': method,
            'Attribute Order': attribute_order,
        }

        for metric in metrics:
            mean = values[(metric, 'mean')]
            std = values[(metric, 'std')]
            n = values[(metric, 'count')]

            se = std / np.sqrt(n)

            row[metric] = (mean, se)

        rows.append(row)

    return pd.DataFrame(rows)


In [ ]:
STRUCTURAL_METRICS = [
    'density',
    'avg_degree',
    'clustering_mean',
    'lcc_ratio',
    'normalized_path_length',
    'louvain_modularity',
]

table_s2 = make_summary_table(
    data,
    STRUCTURAL_METRICS
)

table_s2.head()


In [ ]:
ASSORTATIVITY_METRICS = [
    'gender_assortativity',
    'racethnicity_assortativity',
    'educ5_assortativity',
    'relig_assortativity',
    'vote_assortativity',
]

table_s3 = make_summary_table(
    data,
    ASSORTATIVITY_METRICS
)

table_s3.head()


In [ ]:
def format_mean_se(x, decimals=3):
    mean, se = x

    if pd.isna(mean):
        return '--'

    return f"{mean:.{decimals}f} ({se:.{decimals}f})"


In [ ]:
s2 = table_s2.copy()

S2_LABELS = {
    'density': 'Density',
    'avg_degree': 'Average degree',
    'clustering_mean': 'Clustering',
    'lcc_ratio': 'LCC',
    'normalized_path_length': 'Normalized path',
    'louvain_modularity': 'Modularity',
}

for col in STRUCTURAL_METRICS:
    s2[col] = s2[col].apply(format_mean_se)

s2 = s2.rename(columns=S2_LABELS)

s2['T'] = s2['T'].map(lambda x: f"{x:.1f}")
s2['Model'] = s2['Model'].astype(str)
s2['Method'] = s2['Method'].astype(str)

display(s2)


In [ ]:
s3 = table_s3.copy()

S3_LABELS = {
    'gender_assortativity': 'Gender',
    'racethnicity_assortativity': 'Race/ethnicity',
    'educ5_assortativity': 'Education',
    'relig_assortativity': 'Religion',
    'vote_assortativity': 'Vote',
}

for col in ASSORTATIVITY_METRICS:
    s3[col] = s3[col].apply(format_mean_se)

s3 = s3.rename(columns=S3_LABELS)

s3['T'] = s3['T'].map(lambda x: f"{x:.1f}")
s3['Model'] = s3['Model'].astype(str)
s3['Method'] = s3['Method'].astype(str)

display(s3)


In [ ]:
latex_s2 = s2.to_latex(
    index=False,
    escape=False,
    column_format='llllcccccc',
    caption=(
        r'\textbf{Structural properties of the LLM-generated networks.} '
        r'Values report the mean across 10 runs and two attribute orderings, '
        r'with standard errors in parentheses.'
    ),
    label='tab:structural_metrics',
)

print(latex_s2)

with open(TABLE_DIR / 'table_S2_structural_metrics_sample_100_v3_mutual.tex', 'w') as f:
    f.write(latex_s2)


In [ ]:
latex_s3 = s3.to_latex(
    index=False,
    escape=False,
    column_format='llllccccc',
    caption=(
        r'\textbf{Attribute assortativity in the LLM-generated networks.} '
        r'Values report the mean across 10 runs and two attribute orderings, '
        r'with standard errors in parentheses.'
    ),
    label='tab:assortativity_metrics',
)

print(latex_s3)

with open(TABLE_DIR / 'table_S3_assortativity_metrics_sample_100_v3_mutual.tex', 'w') as f:
    f.write(latex_s3)


## 4 · Empirical plausibility

Each generated network is evaluated against the equal-study weighted central 95% ranges from notebook 01 under the Mutual projection. Density is retained as a descriptive metric but is not used to classify plausibility. A network is labelled fully plausible only when all five structural metrics fall within their empirical ranges.


In [ ]:
PLAUSIBILITY_METRICS = [
    'avg_degree',
    'clustering_mean',
    'lcc_ratio',
    'normalized_path_length',
    'louvain_modularity',
]

ranges = pd.read_csv(REAL_RANGES_CSV)
real_plausibility_ranges = (
    ranges[
        ranges['sample'].eq('primary_friendship_like')
        & ranges['projection'].eq('Mutual')
        & ranges['weighting'].eq('equal_dataset')
    ]
    .set_index('metric')
    .loc[PLAUSIBILITY_METRICS]
)
if not (
    real_plausibility_ranges['n_networks'].eq(58)
    & real_plausibility_ranges['n_datasets'].eq(13)
).all():
    raise ValueError('Notebook 01 ranges are not the expected 58-network, 13-study benchmark.')

for metric, empirical in real_plausibility_ranges.iterrows():
    micro_df[f'{metric}_plausible'] = micro_df[metric].between(
        empirical['lower_2_5'], empirical['upper_97_5'], inclusive='both',
    )
micro_df['plausible_metric_count'] = micro_df[
    [f'{metric}_plausible' for metric in PLAUSIBILITY_METRICS]
].sum(axis=1)
micro_df['plausible_all_5'] = micro_df['plausible_metric_count'].eq(5)
micro_df.to_csv(METRICS_CSV, index=False)

META_COLS = [
    'model', 'file', 'temp', 'chunked', 'chunk_size', 'run', 'attribute_order',
]
METRIC_COLS = [
    column for column in micro_df.columns
    if column not in META_COLS
    and not column.endswith('_plausible')
    and column not in {'plausible_metric_count', 'plausible_all_5'}
]
observed_df = micro_df[META_COLS].copy()
for metric in METRIC_COLS:
    observed_df[f'{metric}_obs'] = micro_df[metric]

print(
    f"Fully plausible individual networks: {micro_df['plausible_all_5'].sum():,}/"
    f"{len(micro_df):,} ({micro_df['plausible_all_5'].mean():.1%})."
)
display(real_plausibility_ranges[[
    'lower_2_5', 'median', 'upper_97_5', 'n_networks', 'n_datasets',
]].round(4))


### 4.1 · Which individual networks are plausible?

The audit table retains one row per generated network. The five Boolean columns show whether that network falls inside each empirical range; `plausible_all_5` requires all five simultaneously. The complete table is written to CSV, while the notebook displays the fully plausible subset.

In [ ]:
CONFIGURATION_ORDER = ['c1', 'c5', 'c10', 'c25', 'c50']
AUDIT_ORDER_LABELS = {
    'gender_race_education_location_age_religion_vote': 'G-first',
    'vote_religion_age_location_education_race_gender': 'V-first',
}
PASS_COLUMNS = [f'{metric}_plausible' for metric in PLAUSIBILITY_METRICS]

network_plausibility_df = micro_df.copy()
network_plausibility_df['configuration'] = np.where(
    network_plausibility_df['chunked'],
    'c' + network_plausibility_df['chunk_size'].astype(int).astype(str),
    'c1',
)
network_plausibility_df['configuration'] = pd.Categorical(
    network_plausibility_df['configuration'],
    categories=CONFIGURATION_ORDER,
    ordered=True,
)
network_plausibility_df['order'] = (
    network_plausibility_df['attribute_order'].map(AUDIT_ORDER_LABELS)
)

audit_columns = [
    'model', 'configuration', 'temp', 'order', 'run', 'file',
    *PASS_COLUMNS, 'plausible_metric_count', 'plausible_all_5',
]
network_plausibility_df = (
    network_plausibility_df[audit_columns]
    .sort_values(['model', 'configuration', 'temp', 'order', 'run', 'file'])
    .reset_index(drop=True)
)
network_plausibility_df.to_csv(PLAUSIBILITY_CSV, index=False)

fully_plausible_networks_df = network_plausibility_df[
    network_plausibility_df['plausible_all_5']
].reset_index(drop=True)

print(
    f'Fully plausible networks: {len(fully_plausible_networks_df):,}/'
    f'{len(network_plausibility_df):,}. Full audit -> {PLAUSIBILITY_CSV}'
)
display(fully_plausible_networks_df)

### 4.2 · Plausibility by model and generation design

These tables are descriptive evaluations, not a configuration-selection procedure. Each table reports the number and share of networks passing all five metrics, plus the pass share for every metric separately.

In [ ]:
def summarize_plausibility(data, group_columns, level):
    aggregations = {
        'n_networks': ('file', 'nunique'),
        'n_fully_plausible': ('plausible_all_5', 'sum'),
        'share_fully_plausible': ('plausible_all_5', 'mean'),
        'mean_metrics_plausible': ('plausible_metric_count', 'mean'),
    }
    for metric in PLAUSIBILITY_METRICS:
        aggregations[f'{metric}_pass_share'] = (f'{metric}_plausible', 'mean')
    summary = (
        data.groupby(group_columns, observed=True, dropna=False)
        .agg(**aggregations)
        .reset_index()
    )
    summary.insert(0, 'grouping', level)
    return summary


plausibility_by_model_df = summarize_plausibility(
    network_plausibility_df, ['model'], 'model',
)
plausibility_by_configuration_df = summarize_plausibility(
    network_plausibility_df, ['configuration'], 'configuration',
)
plausibility_by_temperature_df = summarize_plausibility(
    network_plausibility_df, ['temp'], 'temperature',
)
plausibility_by_order_df = summarize_plausibility(
    network_plausibility_df, ['order'], 'attribute_order',
)
plausibility_by_model_configuration_df = summarize_plausibility(
    network_plausibility_df, ['model', 'configuration'], 'model_configuration',
)
plausibility_by_design_cell_df = summarize_plausibility(
    network_plausibility_df,
    ['model', 'configuration', 'temp', 'order'],
    'model_configuration_temperature_order',
)

plausibility_summary_df = pd.concat(
    [
        plausibility_by_model_df,
        plausibility_by_configuration_df,
        plausibility_by_temperature_df,
        plausibility_by_order_df,
        plausibility_by_model_configuration_df,
        plausibility_by_design_cell_df,
    ],
    ignore_index=True,
)
plausibility_summary_df.to_csv(PLAUSIBILITY_SUMMARY_CSV, index=False)

display_columns = [
    'n_networks', 'n_fully_plausible', 'share_fully_plausible',
    'mean_metrics_plausible',
    *[f'{metric}_pass_share' for metric in PLAUSIBILITY_METRICS],
]

print('By model')
display(plausibility_by_model_df.set_index('model')[display_columns].round(3))
print('By configuration')
display(plausibility_by_configuration_df.set_index('configuration')[display_columns].round(3))
print('By temperature')
display(plausibility_by_temperature_df.set_index('temp')[display_columns].round(3))
print('By attribute order')
display(plausibility_by_order_df.set_index('order')[display_columns].round(3))

In [ ]:
# Exact model × configuration × temperature × order cells (10 runs per cell).
design_cell_columns = [
    'model', 'configuration', 'temp', 'order',
    *display_columns,
]
display(
    plausibility_by_design_cell_df[design_cell_columns]
    .sort_values(['model', 'configuration', 'temp', 'order'])
    .reset_index(drop=True)
    .round(3)
)

# Compact count and share matrices make the configuration pattern easy to inspect.
fully_plausible_count_matrix = plausibility_by_design_cell_df.pivot_table(
    index=['model', 'configuration'],
    columns=['temp', 'order'],
    values='n_fully_plausible',
    observed=True,
)
fully_plausible_share_matrix = plausibility_by_design_cell_df.pivot_table(
    index=['model', 'configuration'],
    columns=['temp', 'order'],
    values='share_fully_plausible',
    observed=True,
)

print('Number fully plausible in each 10-run cell')
display(fully_plausible_count_matrix.fillna(0).astype(int))
print('Share fully plausible in each 10-run cell')
display(fully_plausible_share_matrix.round(2))

## 5 · Structural overview


In [ ]:
# -- Shared plot configuration --

MODEL_LABELS = {
    'llama-31-8b-instruct': 'Llama-3.1 8B',
    'meta_llama_31_70b_instruct_awq_int4': 'Llama-3.1 70B AWQ',
    'mistral-7b-instruct-v03': 'Mistral 7B',
    'qwen25-7b-instruct': 'Qwen2.5 7B',
    'qwen25_32b_instruct_awq': 'Qwen2.5 32B AWQ',
    'qwen25_72b_instruct_awq': 'Qwen2.5 72B AWQ',
}

ORDER_LABELS = {
    'gender_race_education_location_age_religion_vote': 'G-first',
    'vote_religion_age_location_education_race_gender': 'V-first',
}

METRIC_LABELS = {
    'density': 'Density (descriptive)',
    'avg_degree': 'Average degree',
    'degree_gini': 'Degree Gini',
    'clustering_mean': 'Average clustering',
    'lcc_ratio': 'Nodes in LCC',
    'avg_path_length': 'Average path length',
    'normalized_path_length': r'Normalized path $\ell/\log(N_{LCC})$',
    'louvain_modularity': 'Modularity',
}

READABLE_STRUCTURAL_METRICS = PLAUSIBILITY_METRICS
READABLE_ASSORT_METRICS = [f'{ATTR_SHORT[a]}_assortativity' for a in HOMOPHILY_ATTRS]
READABLE_HOMOPHILY_METRICS = [f'{ATTR_SHORT[a]}_homophily' for a in HOMOPHILY_ATTRS]
READABLE_POLARIZATION_METRICS = ['vote_assortativity', 'vote_homophily']
READABLE_CORE_METRICS = READABLE_STRUCTURAL_METRICS + READABLE_POLARIZATION_METRICS
READABLE_KEY_METRICS = READABLE_STRUCTURAL_METRICS + READABLE_ASSORT_METRICS + READABLE_HOMOPHILY_METRICS

READABLE_ATTR_LABELS = {
    'gender': 'Gender',
    'racethnicity': 'Race/Ethnicity',
    'educ5': 'Education',
    'vote': 'Vote',
    'relig': 'Religion',
}
READABLE_METRIC_LABELS = {**METRIC_LABELS}
for short, label in READABLE_ATTR_LABELS.items():
    READABLE_METRIC_LABELS[f'{short}_assortativity'] = f'{label} Assortativity'
    READABLE_METRIC_LABELS[f'{short}_homophily'] = f'{label} Homophily Ratio'


def model_label(model):
    return MODEL_LABELS.get(model, str(model))


def order_label(order):
    if pd.isna(order):
        return 'order NA'
    return ORDER_LABELS.get(order, str(order).replace('_', ' '))


def chunk_label(chunked, chunk_size):
    if not bool(chunked) or pd.isna(chunk_size) or int(chunk_size) == 0:
        return 'c1'
    return f'c{int(chunk_size)}'


def temp_label(temp):
    if pd.isna(temp):
        return 'tNA'
    return f't{float(temp):g}'


def method_sort_key(label):
    m = re.search(r'\d+', str(label))
    return (0, int(m.group(0)) if m else 999)


def add_readable_columns(df):
    out = df.copy()
    out['model_label'] = out['model'].map(MODEL_LABELS).fillna(out['model'])
    out['temp_label'] = out['temp'].apply(temp_label)
    out['method_label'] = out.apply(
        lambda row: chunk_label(row['chunked'], row['chunk_size']), axis=1,
    )
    out['attr_label'] = out['attribute_order'].apply(order_label)
    out['row_label'] = out['model_label'] + ' | ' + out['method_label']
    out['col_label'] = out['temp_label'] + ' | ' + out['attr_label']
    return out


def method_rank(label):
    return method_sort_key(label)


def readable_summary_table(df, metrics=READABLE_KEY_METRICS):
    data = add_readable_columns(df)
    value_cols = []
    for metric in metrics:
        col = f'{metric}_obs'
        if col in data.columns:
            value_cols.append(col)
    group_cols = [
        'model', 'model_label', 'temp', 'temp_label', 'method_label',
        'attr_label', 'attribute_order', 'row_label', 'col_label',
    ]
    summary = (
        data
        .groupby(group_cols, observed=True, dropna=False)[value_cols]
        .agg(['mean', 'std', 'count'])
    )
    summary.columns = [f'{metric}_{stat}' for metric, stat in summary.columns]
    summary = summary.reset_index()
    for col in value_cols:
        std_col = f'{col}_std'
        count_col = f'{col}_count'
        if std_col in summary and count_col in summary:
            summary[f'{col}_se'] = summary[std_col] / np.sqrt(summary[count_col])
    summary['_model_rank'] = summary['model'].map({m: i for i, m in enumerate(MODEL_LABELS)}).fillna(99)
    summary['_method_rank'] = summary['method_label'].map(lambda x: method_rank(x)[1])
    summary['_temp_rank'] = summary['temp'].fillna(-1)
    summary['_attr_rank'] = summary['attr_label'].map({'G-first': 0, 'V-first': 1}).fillna(99)
    return summary.sort_values(['_model_rank', '_method_rank', '_temp_rank', '_attr_rank']).reset_index(drop=True)


def compact_condition_table(summary, metrics=READABLE_CORE_METRICS):
    keep = ['model_label', 'temp_label', 'method_label', 'attr_label']
    for metric in metrics:
        col = f'{metric}_obs_mean'
        if col in summary.columns:
            keep.append(col)
    out = summary[keep].copy()
    out = out.rename(columns={
        'model_label': 'model',
        'temp_label': 'temp',
        'method_label': 'method',
        'attr_label': 'order',
        **{f'{m}_obs_mean': READABLE_METRIC_LABELS.get(m, m) for m in metrics},
    })
    return out.round(3)


def readable_metric_title(metric_col):
    label = metric_col
    for raw, pretty in READABLE_METRIC_LABELS.items():
        label = label.replace(f'{raw}_obs_mean', pretty)
    return label


def shared_color_limits(values, center=None):
    vals = pd.Series(values).dropna().astype(float)
    if vals.empty:
        return None, None
    if center is not None:
        bound = max(abs(vals.min() - center), abs(vals.max() - center))
        return center - bound, center + bound
    return vals.min(), vals.max()


def heatmap_annotation(mean_mat, se_mat=None, fmt='.2f', show_se=True):
    if not show_se or se_mat is None:
        return mean_mat
    annot = mean_mat.copy().astype(object)
    for r in mean_mat.index:
        for c in mean_mat.columns:
            mean_val = mean_mat.loc[r, c]
            se_val = se_mat.loc[r, c] if r in se_mat.index and c in se_mat.columns else np.nan
            if pd.isna(mean_val):
                annot.loc[r, c] = ''
            elif pd.isna(se_val):
                annot.loc[r, c] = format(mean_val, fmt.replace('.', ''))
            else:
                annot.loc[r, c] = f"{mean_val:{fmt}}\n±{se_val:{fmt}}"
    return annot


def plot_readable_heatmaps(summary, metric_cols, title, cmap='viridis', center=None, fmt='.2f', show_se=True):
    model_order = [m for m in MODEL_LABELS.values() if m in set(summary['model_label'])]
    extra_models = [m for m in summary['model_label'].dropna().unique() if m not in model_order]
    model_order = model_order + sorted(extra_models)

    for metric_col in metric_cols:
        if metric_col not in summary.columns:
            continue

        se_col = metric_col.replace('_mean', '_se') if metric_col.endswith('_mean') else None
        vmin, vmax = shared_color_limits(summary[metric_col], center=center)
        fig, axes = plt.subplots(
            1, len(model_order),
            figsize=(max(5.4 * len(model_order), 10), 5.1),
            sharey=False,
            squeeze=False,
        )
        axes = axes.flatten()

        for i, (ax, model) in enumerate(zip(axes, model_order)):
            sub = summary[summary['model_label'] == model].copy()
            if sub.empty:
                ax.set_visible(False)
                continue

            mat = sub.pivot_table(
                index='method_label',
                columns='col_label',
                values=metric_col,
                aggfunc='mean',
                observed=True,
            )
            se_mat = None
            if se_col and se_col in sub.columns:
                se_mat = sub.pivot_table(
                    index='method_label',
                    columns='col_label',
                    values=se_col,
                    aggfunc='mean',
                    observed=True,
                )
            row_order = (
                sub[['method_label', '_method_rank']]
                .drop_duplicates()
                .sort_values('_method_rank')['method_label']
            )
            col_order = (
                sub[['col_label', '_temp_rank', '_attr_rank']]
                .drop_duplicates()
                .sort_values(['_temp_rank', '_attr_rank'])['col_label']
            )
            mat = mat.reindex(index=row_order, columns=col_order)
            if se_mat is not None:
                se_mat = se_mat.reindex(index=row_order, columns=col_order)
            annot = heatmap_annotation(mat, se_mat, fmt=fmt, show_se=show_se)

            sns.heatmap(
                mat, ax=ax, cmap=cmap, center=center, vmin=vmin, vmax=vmax,
                annot=annot, fmt='', linewidths=.5, linecolor='white',
                annot_kws={'fontsize': 12 if show_se else 12},
                cbar=(i == len(model_order) - 1),
                cbar_kws={'shrink': .75, 'label': readable_metric_title(metric_col)},
            )
            ax.set_title(model, fontsize=12, fontweight='bold')
            ax.set_xlabel('Temperature | attribute order')
            ax.set_ylabel('Method / chunk size' if i == 0 else '')
            ax.tick_params(axis='x', rotation=35)
            ax.tick_params(axis='y', rotation=0)

        se_note = ' (mean ± SE)' if show_se else ''
        fig.suptitle(f"{title} - {readable_metric_title(metric_col)}{se_note}", fontsize=14, fontweight='bold', y=1.04)
        plt.tight_layout()
        plt.show()


def plot_method_profiles(summary, metric_cols, title):
    profile = (
        summary
        .groupby(['temp_label', 'method_label', 'attr_label'], observed=True)[metric_cols]
        .mean()
        .reset_index()
    )
    methods = sorted(profile['method_label'].dropna().unique(), key=method_sort_key)
    profile['method_label'] = pd.Categorical(profile['method_label'], categories=methods, ordered=True)
    n = len(metric_cols)
    ncols = min(3, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5.2, nrows * 4.0), squeeze=False)
    axes = axes.flatten()
    for ax, metric_col in zip(axes, metric_cols):
        sns.lineplot(
            data=profile,
            x='method_label', y=metric_col,
            hue='temp_label', style='attr_label', markers=True,
            errorbar=None, ax=ax,
        )
        label = metric_col
        for raw, pretty in READABLE_METRIC_LABELS.items():
            label = label.replace(f'{raw}_obs_mean', pretty)
        ax.set_title(label, fontsize=10, fontweight='bold')
        ax.set_xlabel('Method / chunk size')
        ax.set_ylabel('Mean across models and runs')
        ax.tick_params(axis='x', rotation=0)
        ax.spines[['top', 'right']].set_visible(False)
        ax.legend(fontsize=8, title='Temp / order')
    for ax in axes[n:]:
        ax.set_visible(False)
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


In [ ]:
# 5.1 Compact condition-level table for the five plausibility metrics
readable_summary = readable_summary_table(observed_df)
compact_condition_table(readable_summary).head(12)


In [ ]:
# 5.2 Structural metric heatmaps: rows are model/method, columns are temp/order
plot_readable_heatmaps(
    readable_summary,
    [f'{m}_obs_mean' for m in READABLE_STRUCTURAL_METRICS],
    title='Structural Metrics - Observed Values',
    cmap='YlGnBu',
)


In [ ]:
# 5.3 Method profiles make the chunk-size gradient easier to read
plot_method_profiles(
    readable_summary,
    [f'{m}_obs_mean' for m in READABLE_STRUCTURAL_METRICS],
    title='Structural Metrics - Method Profiles',
)


## 6 · Attribute mixing


In [ ]:
# 6.1 Assortativity heatmaps for every attribute
# Positive values mean connected nodes are more similar on that attribute than expected.
plot_readable_heatmaps(
    readable_summary,
    [f'{m}_obs_mean' for m in READABLE_ASSORT_METRICS],
    title='Attribute Assortativity - Observed Values',
    cmap='RdBu_r', center=0, fmt='.2f', show_se=True,
)


In [ ]:
# 6.2 Homophily-ratio heatmaps for every attribute
# Values below 1 mean fewer cross-group ties than expected: stronger homophily.
plot_readable_heatmaps(
    readable_summary,
    [f'{m}_obs_mean' for m in READABLE_HOMOPHILY_METRICS],
    title='Attribute Homophily Ratio - Observed Values',
    cmap='RdBu_r', center=1, fmt='.2f', show_se=True,
)


## 7 · Main-text figure: network metrics across models and configurations

These two figures adapt Chang et al. (2025, Fig. 3): faint markers are individual generated networks, while the larger independent markers show the mean $\pm$ SE. Temperature 0.1 and 0.7 are plotted separately. The x-axis compares generation methods (`c1`--`c50`), separated by additional spacing and vertical guides; color identifies the five models retained in the manuscript, and raw-point shape identifies persona-attribute order.

Panel a reports the five plausibility metrics—average degree, clustering, the proportion of nodes in the largest connected component, normalized average shortest-path length, and Louvain modularity—against the empirical central 95% ranges from notebook 01. The shaded intervals show the ranges without median reference lines. Density remains available in the supplementary table but is not used for plausibility. Panel b reports nominal assortativity for gender, race/ethnicity, education, religion, and 2020 vote. The 32B Qwen model is intentionally excluded to match the main-text model set; change `MAIN_FIG_MODELS` below to include it.


In [ ]:
# Publication figure for Results Section 2.1
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

MAIN_FIG_MODELS = [
    'llama-31-8b-instruct',
    'mistral-7b-instruct-v03',
    'qwen25-7b-instruct',
    'meta_llama_31_70b_instruct_awq_int4',
    'qwen25_72b_instruct_awq',
]
MAIN_FIG_METHODS = ['c1', 'c5', 'c10', 'c25', 'c50']

MAIN_FIG_MODEL_COLORS = {
    'llama-31-8b-instruct':    '#793B56',  # deep berry
    'mistral-7b-instruct-v03': '#B45791',  # mauve
    'qwen25-7b-instruct':      '#EF895F',  # coral
    'meta_llama_31_70b_instruct_awq_int4': '#86CEB0',  # mint
    'qwen25_72b_instruct_awq': '#078BA8',  # teal-blue
}
MAIN_FIG_MODEL_LABELS = {
    'llama-31-8b-instruct':    'Llama-3.1 8B',
    'mistral-7b-instruct-v03': 'Mistral 7B',
    'qwen25-7b-instruct':      'Qwen2.5 7B',
    'meta_llama_31_70b_instruct_awq_int4': 'Llama-3.1 70B AWQ',
    'qwen25_72b_instruct_awq': 'Qwen2.5 72B AWQ',
}

STRUCTURAL_MAIN = {
    'avg_degree_obs':              'Average degree',
    'clustering_mean_obs':         'Average clustering',
    'lcc_ratio_obs':               'Nodes in LCC',
    'normalized_path_length_obs':  'Average shortest path\n(normalized)',
    'louvain_modularity_obs':      'Modularity',
}
ASSORTATIVITY_MAIN = {
    'gender_assortativity_obs':        'Gender',
    'racethnicity_assortativity_obs':  'Race / ethnicity',
    'educ5_assortativity_obs':         'Education',
    'relig_assortativity_obs':         'Religion',
    'vote_assortativity_obs':          '2020 vote',
}

# Attribute order is phrased in terms of vote position to match the manuscript.
RAW_ORDER_MARKERS = {
    'G-first': ('o', 'vote-last'),
    'V-first': ('^', 'vote-first'),
}


def build_main_figure_data(df):
    data = add_readable_columns(df)
    data = data[
        data['model'].isin(MAIN_FIG_MODELS)
        & data['method_label'].isin(MAIN_FIG_METHODS)
    ].copy()
    data['method_label'] = pd.Categorical(
        data['method_label'], categories=MAIN_FIG_METHODS, ordered=True
    )
    data['model'] = pd.Categorical(
        data['model'], categories=MAIN_FIG_MODELS, ordered=True
    )
    return data


def main_figure_summary(data):
    metric_map = {**STRUCTURAL_MAIN, **ASSORTATIVITY_MAIN}
    long = data.melt(
        id_vars=['model', 'model_label', 'method_label', 'temp', 'temp_label'],
        value_vars=list(metric_map),
        var_name='metric', value_name='value',
    ).dropna(subset=['value'])
    out = (
        long.groupby(
            ['model', 'model_label', 'method_label', 'temp', 'temp_label', 'metric'],
            observed=True, dropna=False,
        )['value']
        .agg(mean='mean', std='std', n='count')
        .reset_index()
    )
    out['se'] = out['std'] / np.sqrt(out['n'])
    out['section'] = np.where(
        out['metric'].isin(STRUCTURAL_MAIN), 'Structural properties', 'Assortativity'
    )
    out['metric_label'] = out['metric'].map(metric_map)
    return out[[
        'section', 'metric', 'metric_label', 'model', 'model_label', 'temp', 'temp_label',
        'method_label', 'mean', 'se', 'std', 'n',
    ]]


def _plot_metric_axis(ax, data, summary, metric, title, temp, rng, assortativity=False):
    x = np.arange(len(MAIN_FIG_METHODS), dtype=float) * 1.45
    offsets = np.linspace(-0.36, 0.36, len(MAIN_FIG_MODELS))
    empirical_metric = metric.removesuffix('_obs')
    if not assortativity and empirical_metric in real_plausibility_ranges.index:
        empirical = real_plausibility_ranges.loc[empirical_metric]
        ax.axhspan(
            empirical['lower_2_5'], empirical['upper_97_5'],
            color='#BDBDBD', alpha=.25, zorder=0,
        )
    for boundary in (x[:-1] + x[1:]) / 2:
        ax.axvline(boundary, color='#C9C9C9', linewidth=0.9, zorder=0)

    # Raw-network layer: 10 runs for each attribute-order cell at this temperature.
    for model_i, model in enumerate(MAIN_FIG_MODELS):
        model_color = MAIN_FIG_MODEL_COLORS[model]
        for method_i, method in enumerate(MAIN_FIG_METHODS):
            base = x[method_i] + offsets[model_i]
            for order, (marker, _) in RAW_ORDER_MARKERS.items():
                values = data.loc[
                    (data['model'] == model)
                    & (data['method_label'] == method)
                    & (data['attr_label'] == order),
                    metric,
                ].dropna().to_numpy()
                if not len(values):
                    continue
                jitter = rng.uniform(-0.035, 0.035, size=len(values))
                ax.scatter(
                    np.full(len(values), base) + jitter, values,
                    s=18, marker=marker, color=model_color, alpha=0.22,
                    linewidths=0, rasterized=True, zorder=1,
                )

        means, ses = [], []
        for method in MAIN_FIG_METHODS:
            row = summary.loc[
                (summary['model'] == model)
                & (summary['method_label'] == method)
                & np.isclose(summary['temp'].astype(float), temp)
                & (summary['metric'] == metric)
            ]
            means.append(row['mean'].iloc[0] if len(row) else np.nan)
            ses.append(row['se'].iloc[0] if len(row) else np.nan)
        xpos = x + offsets[model_i]
        ax.errorbar(
            xpos, means, yerr=ses, fmt='o', linestyle='none', markersize=7.2,
            color=model_color, markeredgecolor='white', markeredgewidth=0.75,
            elinewidth=1.5, capsize=2.8, capthick=1.25, zorder=4,
        )

    ax.set_title(title, fontsize=20, fontweight='normal', pad=11)
    ax.set_xticks(x, MAIN_FIG_METHODS)
    ax.tick_params(axis='both', labelsize=18, length=4)
    ax.grid(axis='y', color='#D9D9D9', linewidth=0.75, alpha=0.75)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_color('#777777')
    ax.spines[['left', 'bottom']].set_linewidth(0.65)

    if assortativity:
        ax.axhline(0, color='#555555', linestyle=':', linewidth=0.8, zorder=2)
    elif metric not in {'normalized_path_length_obs', 'avg_degree_obs'}:
        ax.set_ylim(-0.02, 1.02)
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])


def plot_main_network_metrics(data, temp, output_stem):
    plot_data = data[np.isclose(data['temp'].astype(float), temp)].copy()
    summary = main_figure_summary(plot_data)
    fig, axes = plt.subplots(2, 5, figsize=(19.0, 9.4), constrained_layout=False)
    rng = np.random.default_rng(RANDOM_SEED)

    for ax, (metric, label) in zip(axes[0], STRUCTURAL_MAIN.items()):
        _plot_metric_axis(ax, plot_data, summary, metric, label, temp, rng, assortativity=False)
    for ax, (metric, label) in zip(axes[1], ASSORTATIVITY_MAIN.items()):
        _plot_metric_axis(ax, plot_data, summary, metric, label, temp, rng, assortativity=True)
    for ax in axes[0, len(STRUCTURAL_MAIN):]:
        ax.set_visible(False)
    for ax in axes[1, len(ASSORTATIVITY_MAIN):]:
        ax.set_visible(False)

    # Use one common scale for all assortativity panels.
    assort_values = plot_data[list(ASSORTATIVITY_MAIN)].to_numpy(dtype=float).ravel()
    assort_values = assort_values[np.isfinite(assort_values)]
    a_min = min(0, np.quantile(assort_values, 0.005))
    a_max = max(0, np.quantile(assort_values, 0.995))
    a_pad = max((a_max - a_min) * 0.08, 0.03)
    for ax in axes[1, :len(ASSORTATIVITY_MAIN)]:
        ax.set_ylim(a_min - a_pad, a_max + a_pad)
        ax.yaxis.set_major_locator(mticker.MaxNLocator(5))

    axes[0, 0].set_ylabel('Metric value', fontsize=20)
    axes[1, 0].set_ylabel('Assortativity', fontsize=20)
    fig.text(0.008, 0.905, 'a', fontsize=34, fontweight='normal')
    fig.text(0.008, 0.475, 'b', fontsize=34, fontweight='normal')
    # fig.text(0.04, 0.905, 'Structural properties', fontsize=22, fontweight='normal')
    # fig.text(0.04, 0.475, 'Attribute assortativity', fontsize=22, fontweight='normal')
    fig.text(0.985, 0.955, f'T={temp:g}', ha='right', fontsize=22, fontweight='normal')

    model_handles = [
        Line2D(
            [0], [0], color=MAIN_FIG_MODEL_COLORS[m], marker='o', markersize=8,
            linestyle='none', label=MAIN_FIG_MODEL_LABELS[m],
        )
        for m in MAIN_FIG_MODELS
    ]
    condition_handles = [
        Line2D(
            [0], [0], color='#777777', marker=marker, linestyle='', markersize=7,
            label=label, alpha=0.75,
        )
        for marker, label in RAW_ORDER_MARKERS.values()
    ]
    empirical_handles = [
        Patch(facecolor='#BDBDBD', alpha=.25, label='Real-network central 95%'),
    ]
    fig.legend(
        handles=model_handles + condition_handles + empirical_handles,
        loc='lower center', bbox_to_anchor=(0.5, 0.012),
        ncol=5, frameon=False, fontsize=16, handletextpad=0.45, columnspacing=1.5,
    )

    fig.subplots_adjust(left=0.07, right=0.995, top=0.86, bottom=0.18, hspace=0.55, wspace=0.34)
    fig.savefig(f'{output_stem}.pdf', bbox_inches='tight')
    fig.savefig(f'{output_stem}.png', dpi=300, bbox_inches='tight')
    summary.to_csv(f'{output_stem}_summary.csv', index=False)
    return fig, summary


main_figure_data = build_main_figure_data(observed_df)
fig_network_metrics_t01, network_metrics_summary_t01 = plot_main_network_metrics(
    main_figure_data, temp=0.1, output_stem=FIGURE_DIR / 'Fig_network_metrics_T01_sample_100_v3_mutual'
)
fig_network_metrics_t07, network_metrics_summary_t07 = plot_main_network_metrics(
    main_figure_data, temp=0.7, output_stem=FIGURE_DIR / 'Fig_network_metrics_T07_sample_100_v3_mutual'
)
fig_network_metrics_t01.savefig(FIGURE_DIR / 'Fig_2_sample_v3_mutual_01.pdf', bbox_inches='tight')
paper_figure_path = PAPER_IMAGE_DIR / 'Fig_2_network_plausibility_T01.pdf'
fig_network_metrics_t01.savefig(paper_figure_path, bbox_inches='tight')
fig_network_metrics_t07.savefig(FIGURE_DIR / 'Fig_2_sample_v3_mutual_07.pdf', bbox_inches='tight')
network_metrics_summary = pd.concat(
    [network_metrics_summary_t01, network_metrics_summary_t07], ignore_index=True
)
network_metrics_summary.to_csv(DATA_DIR / 'Fig_network_metrics_summary_by_temp_sample_100_v3_mutual.csv', index=False)
print(
    f'Saved paper-ready T=0.1 PDF to {paper_figure_path}; also saved T=0.1 and T=0.7 '
    'figures (PDF and PNG), manuscript PDFs, and summary CSV; '
    f'{len(main_figure_data):,} networks, '
    f'{main_figure_data.model.nunique()} models.'
)
display(network_metrics_summary.head(10).round(4))


## 8 · Degree distributions by generation configuration

Degree distributions are shown in two complementary forms: probability histograms, $P(D=k)$, and complementary cumulative distributions, $P(D \geq k)$. Both are separated by temperature, generation configuration, and attribute order. Within each panel, each model curve pools node degrees over only the ten runs in that exact condition. This preserves the full model × configuration × temperature × attribute-order design. The histograms show the distribution body, while the logarithmic CCDF makes differences in the upper-degree tail visible.


In [ ]:
DEGREE_FIGURE_STEM = FIGURE_DIR / 'Fig_degree_distribution_sample_100_v3_mutual'
DEGREE_HISTOGRAM_STEM = FIGURE_DIR / 'Fig_degree_histogram_sample_100_v3_mutual'
DEGREE_GRID = np.arange(100, dtype=int)  # every generated graph has 100 nodes
DEGREE_ORDER_ROWS = ['G-first', 'V-first']
DEGREE_ORDER_LABELS = {
    'G-first': 'Vote last',
    'V-first': 'Vote first',
}


def build_degree_distribution_data(graphs):
    node_rows = []
    for model, files in graphs.items():
        if model not in MAIN_FIG_MODELS:
            continue
        for fname, G in files.items():
            meta = parse_filename(fname)
            method = (
                'c1' if not meta['chunked'] or meta['chunk_size'] == 0
                else f"c{int(meta['chunk_size'])}"
            )
            if method not in MAIN_FIG_METHODS:
                continue
            attr_label = ORDER_LABELS.get(meta['attribute_order'])
            for _, degree in G.degree():
                node_rows.append({
                    'model': model,
                    'model_label': MAIN_FIG_MODEL_LABELS[model],
                    'temp': float(meta['temp']),
                    'method': method,
                    'attribute_order': meta['attribute_order'],
                    'attr_label': attr_label,
                    'run': int(meta['run']),
                    'file': fname,
                    'degree': int(degree),
                })

    nodes = pd.DataFrame(node_rows)
    group_cols = [
        'model', 'model_label', 'temp', 'method',
        'attribute_order', 'attr_label',
    ]
    distribution_rows = []
    for keys, group in nodes.groupby(group_cols, observed=True, dropna=False):
        counts = np.bincount(
            group['degree'], minlength=len(DEGREE_GRID)
        )[:len(DEGREE_GRID)]
        probability = counts / counts.sum()
        ccdf = counts[::-1].cumsum()[::-1] / counts.sum()
        metadata = dict(zip(group_cols, keys))
        n_graphs = group['file'].nunique()
        n_node_observations = len(group)
        for degree, count, prob, survival in zip(
            DEGREE_GRID, counts, probability, ccdf
        ):
            distribution_rows.append({
                **metadata,
                'degree': int(degree),
                'count': int(count),
                'probability': prob,
                'ccdf': survival,
                'n_graphs': n_graphs,
                'n_node_observations': n_node_observations,
            })
    return nodes, pd.DataFrame(distribution_rows)


def plot_degree_distributions(distributions, temp, output_stem):
    fig, axes = plt.subplots(
        len(DEGREE_ORDER_ROWS), len(MAIN_FIG_METHODS),
        figsize=(21.5, 8.2), sharex=True, sharey=True,
        constrained_layout=False,
    )
    plot_data = distributions[np.isclose(distributions['temp'], temp)]

    for row_i, order in enumerate(DEGREE_ORDER_ROWS):
        for col_i, method in enumerate(MAIN_FIG_METHODS):
            ax = axes[row_i, col_i]
            for model in MAIN_FIG_MODELS:
                curve = plot_data[
                    (plot_data['attr_label'] == order)
                    & (plot_data['method'] == method)
                    & (plot_data['model'] == model)
                    & (plot_data['ccdf'] > 0)
                ].sort_values('degree')
                if curve.empty:
                    continue
                ax.step(
                    curve['degree'], curve['ccdf'], where='post',
                    color=MAIN_FIG_MODEL_COLORS[model],
                    linewidth=1.8, alpha=.95,
                )

            if row_i == 0:
                ax.set_title(method, fontsize=17, pad=9)
            if col_i == 0:
                ax.set_ylabel(
                    f"{DEGREE_ORDER_LABELS[order]}\n$P(D \\geq k)$",
                    fontsize=14,
                )
            ax.set_yscale('log')
            ax.set_ylim(8e-4, 1.08)
            ax.set_xlim(0, DEGREE_GRID.max())
            ax.grid(
                True, which='major', color='#D9D9D9',
                linewidth=.7, alpha=.75,
            )
            ax.grid(
                True, which='minor', axis='y', color='#ECECEC',
                linewidth=.45, alpha=.65,
            )
            ax.tick_params(axis='both', labelsize=11, length=3.5)
            ax.spines[['top', 'right']].set_visible(False)
            ax.spines[['left', 'bottom']].set_color('#777777')
            ax.spines[['left', 'bottom']].set_linewidth(.65)

    fig.supxlabel('Degree $k$ (mutual ties)', fontsize=15, y=.105)
    fig.text(.985, .955, f'T={temp:g}', ha='right', fontsize=19)
    model_handles = [
        Line2D(
            [0], [0], color=MAIN_FIG_MODEL_COLORS[model],
            linewidth=2.2, label=MAIN_FIG_MODEL_LABELS[model],
        )
        for model in MAIN_FIG_MODELS
    ]
    fig.legend(
        handles=model_handles, loc='lower center',
        bbox_to_anchor=(.5, .015), ncol=len(model_handles),
        frameon=False, fontsize=13, handlelength=2.5,
        columnspacing=1.7,
    )
    fig.subplots_adjust(
        left=.07, right=.995, top=.89, bottom=.19,
        hspace=.18, wspace=.16,
    )
    fig.savefig(f'{output_stem}.pdf', bbox_inches='tight')
    fig.savefig(f'{output_stem}.png', dpi=300, bbox_inches='tight')
    return fig


def plot_degree_histograms(distributions, temp, output_stem):
    fig, axes = plt.subplots(
        len(DEGREE_ORDER_ROWS), len(MAIN_FIG_METHODS),
        figsize=(21.5, 8.2), sharex=True, sharey=True,
        constrained_layout=False,
    )
    plot_data = distributions[np.isclose(distributions['temp'], temp)]
    bin_edges = np.arange(-.5, len(DEGREE_GRID) + .5, 1)
    y_max = plot_data['probability'].max() * 1.08

    for row_i, order in enumerate(DEGREE_ORDER_ROWS):
        for col_i, method in enumerate(MAIN_FIG_METHODS):
            ax = axes[row_i, col_i]
            for model in MAIN_FIG_MODELS:
                histogram = plot_data[
                    (plot_data['attr_label'] == order)
                    & (plot_data['method'] == method)
                    & (plot_data['model'] == model)
                ].sort_values('degree')
                if histogram.empty:
                    continue
                ax.stairs(
                    histogram['probability'], bin_edges, baseline=None,
                    color=MAIN_FIG_MODEL_COLORS[model],
                    linewidth=1.55, alpha=.92,
                )

            if row_i == 0:
                ax.set_title(method, fontsize=17, pad=9)
            if col_i == 0:
                ax.set_ylabel(
                    f"{DEGREE_ORDER_LABELS[order]}\n$P(D=k)$",
                    fontsize=14,
                )
            ax.set_ylim(0, y_max)
            ax.set_xlim(-.5, DEGREE_GRID.max() + .5)
            ax.grid(
                True, axis='y', color='#D9D9D9',
                linewidth=.7, alpha=.75,
            )
            ax.tick_params(axis='both', labelsize=11, length=3.5)
            ax.spines[['top', 'right']].set_visible(False)
            ax.spines[['left', 'bottom']].set_color('#777777')
            ax.spines[['left', 'bottom']].set_linewidth(.65)

    fig.supxlabel('Degree $k$ (mutual ties)', fontsize=15, y=.105)
    fig.text(.985, .955, f'T={temp:g}', ha='right', fontsize=19)
    model_handles = [
        Line2D(
            [0], [0], color=MAIN_FIG_MODEL_COLORS[model],
            linewidth=2.2, label=MAIN_FIG_MODEL_LABELS[model],
        )
        for model in MAIN_FIG_MODELS
    ]
    fig.legend(
        handles=model_handles, loc='lower center',
        bbox_to_anchor=(.5, .015), ncol=len(model_handles),
        frameon=False, fontsize=13, handlelength=2.5,
        columnspacing=1.7,
    )
    fig.subplots_adjust(
        left=.07, right=.995, top=.89, bottom=.19,
        hspace=.18, wspace=.16,
    )
    fig.savefig(f'{output_stem}.pdf', bbox_inches='tight')
    fig.savefig(f'{output_stem}.png', dpi=300, bbox_inches='tight')
    return fig


degree_node_df, degree_distribution_df = build_degree_distribution_data(graphs)
degree_distribution_df.to_csv(f'{DEGREE_FIGURE_STEM}.csv', index=False)

fig_degree_t01 = plot_degree_distributions(
    degree_distribution_df, temp=.1,
    output_stem=f'{DEGREE_FIGURE_STEM}_T01',
)
fig_degree_t07 = plot_degree_distributions(
    degree_distribution_df, temp=.7,
    output_stem=f'{DEGREE_FIGURE_STEM}_T07',
)
fig_degree_hist_t01 = plot_degree_histograms(
    degree_distribution_df, temp=.1,
    output_stem=f'{DEGREE_HISTOGRAM_STEM}_T01',
)
fig_degree_hist_t07 = plot_degree_histograms(
    degree_distribution_df, temp=.7,
    output_stem=f'{DEGREE_HISTOGRAM_STEM}_T07',
)

degree_cell_counts = (
    degree_distribution_df
    .groupby(
        ['model_label', 'temp', 'method', 'attr_label'],
        observed=True,
    )[['n_graphs', 'n_node_observations']]
    .first()
    .reset_index()
)
print(
    f"Saved degree-distribution CSV, CCDF figures, and histogram figures "
    f"for T=0.1 and T=0.7; "
    f"{len(degree_cell_counts)} exact condition cells and "
    f"{len(degree_node_df):,} node-degree observations."
)
display(degree_cell_counts.head(10))


## 9 · Real versus LLM-generated degree distributions

This comparison uses normalized degree, $d_i/(N-1)$, for both empirical and generated networks. The empirical reference is the Mutual projection and gives equal weight first to each network within a dataset and then to each of the 13 datasets. LLM-generated distributions remain separated by model, generation configuration, temperature, and attribute order; each exact condition pools its ten 100-node runs. Each temperature figure contains both probability histograms and logarithmic CCDFs.


In [ ]:
REAL_LLM_DEGREE_STEM = FIGURE_DIR / 'Fig_normalized_degree_comparison_real_vs_llm'
REAL_POOLED_DEGREE_CSV = FIGURE_DIR / 'Fig_normalized_degree_histogram_real_networks_pooled.csv'
COMPARISON_DEGREE_BINS = np.linspace(0, 1, 51)
COMPARISON_HISTOGRAM_BINS = np.linspace(0, 1, 26)  # coarser display only
COMPARISON_BIN_CENTERS = (
    COMPARISON_DEGREE_BINS[:-1] + COMPARISON_DEGREE_BINS[1:]
) / 2


def coarsen_comparison_histogram(probability):
    probability = np.asarray(probability, dtype=float)
    if len(probability) != 50:
        raise ValueError('Expected the shared 50-bin comparison grid.')
    return probability.reshape(25, 2).sum(axis=1)


def build_llm_normalized_degree_data(nodes):
    group_cols = [
        'model', 'model_label', 'temp', 'method',
        'attribute_order', 'attr_label',
    ]
    rows = []
    for keys, group in nodes.groupby(group_cols, observed=True, dropna=False):
        normalized_degree = group['degree'].to_numpy(dtype=float) / 99
        counts, _ = np.histogram(normalized_degree, bins=COMPARISON_DEGREE_BINS)
        probability = counts / counts.sum()
        ccdf = probability[::-1].cumsum()[::-1]
        metadata = dict(zip(group_cols, keys))
        for bin_index, (left, right, center, count, prob, survival) in enumerate(zip(
            COMPARISON_DEGREE_BINS[:-1], COMPARISON_DEGREE_BINS[1:],
            COMPARISON_BIN_CENTERS, counts, probability, ccdf,
        )):
            rows.append({
                'source': 'LLM-generated',
                **metadata,
                'bin_index': bin_index,
                'normalized_degree_left': left,
                'normalized_degree_right': right,
                'normalized_degree_center': center,
                'count': int(count),
                'probability': prob,
                'ccdf': survival,
                'n_graphs': group['file'].nunique(),
                'n_node_observations': len(group),
                'n_datasets': np.nan,
            })
    return pd.DataFrame(rows)


def load_real_normalized_degree_reference(path=REAL_POOLED_DEGREE_CSV):
    real = pd.read_csv(path)
    real = real[real['projection'] == 'Mutual'].sort_values('bin_index').copy()
    if len(real) != len(COMPARISON_BIN_CENTERS):
        raise ValueError('Real and LLM comparison grids have different lengths.')
    if not np.allclose(real['normalized_degree_center'], COMPARISON_BIN_CENTERS):
        raise ValueError('Real and LLM normalized-degree bins do not align.')
    real['ccdf'] = real['probability'].to_numpy()[::-1].cumsum()[::-1]
    return real


def comparison_export(llm_distributions, real_reference):
    real = real_reference.copy()
    real['source'] = 'Real networks'
    real['model'] = 'real_networks'
    real['model_label'] = 'Real networks'
    real['temp'] = np.nan
    real['method'] = 'all'
    real['attribute_order'] = 'all'
    real['attr_label'] = 'all'
    real['count'] = np.nan
    real['n_graphs'] = real['n_networks']
    real['n_node_observations'] = np.nan
    columns = [
        'source', 'model', 'model_label', 'temp', 'method',
        'attribute_order', 'attr_label', 'bin_index',
        'normalized_degree_left', 'normalized_degree_right',
        'normalized_degree_center', 'count', 'probability', 'ccdf',
        'n_graphs', 'n_node_observations', 'n_datasets',
    ]
    return pd.concat(
        [llm_distributions[columns], real[columns]], ignore_index=True,
    )


def plot_real_vs_llm_degree(llm_distributions, real_reference, temp, output_stem):
    row_specs = [
        ('G-first', 'probability'), ('G-first', 'ccdf'),
        ('V-first', 'probability'), ('V-first', 'ccdf'),
    ]
    fig, axes = plt.subplots(
        len(row_specs), len(MAIN_FIG_METHODS),
        figsize=(21.5, 13.4), sharex=True, constrained_layout=False,
    )
    plot_data = llm_distributions[np.isclose(llm_distributions['temp'], temp)]
    real_probability = real_reference['probability'].to_numpy()
    real_histogram_probability = coarsen_comparison_histogram(real_probability)
    real_ccdf = real_reference['ccdf'].to_numpy()
    llm_histogram_max = max(
        coarsen_comparison_histogram(group.sort_values('bin_index')['probability']).max()
        for _, group in plot_data.groupby(
            ['model', 'method', 'attr_label'], observed=True,
        )
    )
    hist_ymax = max(llm_histogram_max, real_histogram_probability.max()) * 1.06

    for row_i, (order, statistic) in enumerate(row_specs):
        for col_i, method in enumerate(MAIN_FIG_METHODS):
            ax = axes[row_i, col_i]
            for model in MAIN_FIG_MODELS:
                curve = plot_data[
                    (plot_data['attr_label'] == order)
                    & (plot_data['method'] == method)
                    & (plot_data['model'] == model)
                ].sort_values('bin_index')
                if curve.empty:
                    continue
                if statistic == 'probability':
                    ax.stairs(
                        coarsen_comparison_histogram(curve['probability']),
                        COMPARISON_HISTOGRAM_BINS,
                        baseline=None, color=MAIN_FIG_MODEL_COLORS[model],
                        linewidth=1.35, alpha=.88,
                    )
                else:
                    positive = curve['ccdf'] > 0
                    ax.step(
                        curve.loc[positive, 'normalized_degree_left'],
                        curve.loc[positive, 'ccdf'], where='post',
                        color=MAIN_FIG_MODEL_COLORS[model],
                        linewidth=1.55, alpha=.90,
                    )

            if statistic == 'probability':
                ax.stairs(
                    real_histogram_probability, COMPARISON_HISTOGRAM_BINS, baseline=None,
                    color='#111111', linewidth=2.4, linestyle='--', alpha=.98,
                )
                ax.set_ylim(0, hist_ymax)
            else:
                positive = real_ccdf > 0
                ax.step(
                    real_reference.loc[positive, 'normalized_degree_left'],
                    real_ccdf[positive], where='post', color='#111111',
                    linewidth=2.4, linestyle='--', alpha=.98,
                )
                ax.set_yscale('log')
                ax.set_ylim(8e-4, 1.08)

            if row_i == 0:
                ax.set_title(method, fontsize=16, pad=8)
            if col_i == 0:
                statistic_label = (
                    'Probability per bin' if statistic == 'probability'
                    else '$P(d/(N-1) \geq x)$'
                )
                ax.set_ylabel(
                    f"{DEGREE_ORDER_LABELS[order]}\n{statistic_label}", fontsize=12.5,
                )
            ax.set_xlim(0, 1)
            ax.set_xticks([0, .25, .5, .75, 1])
            ax.grid(True, which='major', color='#D9D9D9', linewidth=.65, alpha=.72)
            if statistic == 'ccdf':
                ax.grid(
                    True, which='minor', axis='y', color='#ECECEC',
                    linewidth=.4, alpha=.55,
                )
            ax.tick_params(axis='both', labelsize=10, length=3.5)
            ax.spines[['top', 'right']].set_visible(False)
            ax.spines[['left', 'bottom']].set_color('#777777')
            ax.spines[['left', 'bottom']].set_linewidth(.65)

    fig.supxlabel('Normalized degree $d/(N-1)$', fontsize=15, y=.075)
    fig.text(.985, .975, f'T={temp:g}', ha='right', fontsize=18)
    handles = [
        Line2D(
            [0], [0], color=MAIN_FIG_MODEL_COLORS[model], linewidth=2.1,
            label=MAIN_FIG_MODEL_LABELS[model],
        )
        for model in MAIN_FIG_MODELS
    ] + [
        Line2D(
            [0], [0], color='#111111', linewidth=2.5, linestyle='--',
            label='Real networks (Mutual; equal dataset weight)',
        )
    ]
    fig.legend(
        handles=handles, loc='lower center', bbox_to_anchor=(.5, .012),
        ncol=3, frameon=False, fontsize=11.5, handlelength=2.7,
        columnspacing=1.6,
    )
    fig.subplots_adjust(
        left=.075, right=.995, top=.94, bottom=.14, hspace=.25, wspace=.17,
    )
    fig.savefig(f'{output_stem}.pdf', bbox_inches='tight')
    fig.savefig(f'{output_stem}.png', dpi=300, bbox_inches='tight')
    return fig


llm_normalized_degree_df = build_llm_normalized_degree_data(degree_node_df)
real_normalized_degree_reference = load_real_normalized_degree_reference()
real_llm_degree_comparison_df = comparison_export(
    llm_normalized_degree_df, real_normalized_degree_reference,
)
real_llm_degree_comparison_df.to_csv(f'{REAL_LLM_DEGREE_STEM}.csv', index=False)

fig_real_llm_degree_t01 = plot_real_vs_llm_degree(
    llm_normalized_degree_df, real_normalized_degree_reference, temp=.1,
    output_stem=f'{REAL_LLM_DEGREE_STEM}_T01',
)
fig_real_llm_degree_t07 = plot_real_vs_llm_degree(
    llm_normalized_degree_df, real_normalized_degree_reference, temp=.7,
    output_stem=f'{REAL_LLM_DEGREE_STEM}_T07',
)

comparison_probability_checks = (
    llm_normalized_degree_df
    .groupby(['model', 'temp', 'method', 'attr_label'], observed=True)['probability']
    .sum()
)
print(
    f'Saved real-versus-LLM normalized-degree comparison CSV and figures; '
    f'{comparison_probability_checks.size} exact LLM condition cells. '
    f'LLM probability sums: {comparison_probability_checks.min():.6f}--'
    f'{comparison_probability_checks.max():.6f}; real reference: '
    f"{real_normalized_degree_reference['probability'].sum():.6f}."
)
